# MRIxFields Stage 2 v2 — NN coupling + identity-start + correct decode

Re-run of the Etapa-2 transport with the three corrections the held-out 0009 gate forced.
What that gate showed, in one line: v1 learned a per-field intensity rescaling and nothing
structural — it beat identity on nRMSE only on the 13 of 60 pairs where the intensity
scales are wildly mismatched, and lost on the other 47, with SSIM flat at 0.8755 against a
0.9573 ceiling.

Changes carried by the pinned commit:

1. `coupling: nn` — global nearest-neighbour retrieval over the whole field pool, instead of
   a Hungarian assignment inside a batch of 8 drawn from pools of 37-191.
2. `zero_init_output` — the untrained transport IS the identity baseline, so the run cannot
   start below the thing it must beat.
3. Full-volume decode, and the gate no longer inherits the *encode* halo (v1 silently decoded
   at a 4-latent-voxel halo, about half the decoder's receptive field).

**Subject 0009 stays frozen.** It was spent once. This notebook gates on the training
travellers 0006/0007, which is a development signal, not held-out evidence.

**2026-07-29 update — traveller leakage fix.** The official data description gives 40
travellers total: 3 in our Training pool (0006, 0007, 0009), 0 in Validation. With
`coupling: nn`, a traveller left in train retrieves its OWN target-field volume as the
globally nearest neighbour — same anatomy is trivially nearest — so it gets real paired
supervision and any score on it measures memorization, not generalization. This notebook now
resplits 0006 into validation (1-1-1 with 0007 train / 0009 test) before training, and the
CLI refuses to score a training-split subject unless `--allow-training-subjects` is passed
explicitly. **0009 remains frozen and untouched.**


## 1. Runtime and pinned repository

In [ ]:
!nvidia-smi
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable an A100 GPU runtime before continuing."
print("GPU:", torch.cuda.get_device_name(0))
assert "A100" in torch.cuda.get_device_name(0), "This full-volume protocol expects an A100."

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO = Path("/content/MRIxFields")
EXPECTED_REPO_SHA = "34496bb6f43c1957d4f9ced527bbc9be662b7d74"

if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    ["git", "clone", "https://github.com/GuillermoTafoya/MRIxFields.git", str(REPO)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_REPO_SHA],
    check=True,
)
head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True,
).strip()
print("HEAD:", head)
assert head == EXPECTED_REPO_SHA

%cd /content/MRIxFields
!pip -q install -e ".[nifti,evaluation,official-evaluation]" scipy

for command in (
    "build-latent-bank",
    "train-stage2-transport",
    "eval-stage2-transport",
):
    subprocess.run(
        ["python", "-m", "fieldbridge.cli", command, "--help"],
        check=True,
        stdout=subprocess.DEVNULL,
    )
print("Repository and CLI commands verified.")

## 2. Drive paths and immutable identities

In [ ]:
import hashlib
import json
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/MRIxFields2026"
DATA_ROOT = f"{DRIVE}/Data"
CKPT = f"{DRIVE}/vae_kl_vae_best.pt"
SPLIT_SRC = f"{DRIVE}/split_v3.json"
OLD_DATA_ROOT = r"D:\MRI_Field_2026\Data"

# New paths: the existing tiled bank remains untouched.
LATENTS = f"{DRIVE}/LatentBanks/runC_ep015_step047700_74132b9c_full_bf16"
WORK = f"{DRIVE}/Runs/stage2_gate_runC_74132b9c_c4b9c39_full"
SPLIT = f"{WORK}/split_colab.json"
VAE_CONFIG = "configs/experiment/stage1_vae_v2_fgw_freebits.yaml"

EXPECTED_CKPT_SHA = (
    "74132b9c514bb91b86d8eb43c63542780"
    "bce11304e31e67d3bf75c90ff5d4d79"
)
EXPECTED_SPLIT_SHA = (
    "f6a19d7a31c4c3bb73edd92088ea07819"
    "2e88ee4b276309bad81c548ab7f94d5"
)
EXPECTED_COUNTS = {"train": 1590, "validation": 189, "test": 205}

Path(WORK).mkdir(parents=True, exist_ok=True)
os.environ["MRIXFIELDS_DATA_ROOT"] = DATA_ROOT
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for path in (CKPT, SPLIT_SRC):
    assert Path(path).is_file(), f"Missing required Drive file: {path}"
assert Path(DATA_ROOT).is_dir(), f"Missing official data directory: {DATA_ROOT}"

checkpoint_sha = sha256(CKPT)
split_sha = sha256(SPLIT_SRC)
print("checkpoint SHA256:", checkpoint_sha)
print("split SHA256:", split_sha)
assert checkpoint_sha == EXPECTED_CKPT_SHA, "STOP: wrong Run C checkpoint."
assert split_sha == EXPECTED_SPLIT_SHA, "STOP: wrong frozen split."

# A sentinel prevents a later Run-all from mixing another encoding protocol
# into this resumable directory.
bank_dir = Path(LATENTS)
sentinel = bank_dir / "_full_bank_request.json"
request = {
    "contract": "runC-full-volume-latent-bank-v1",
    "repository_sha": EXPECTED_REPO_SHA,
    "vae_checkpoint_sha256": EXPECTED_CKPT_SHA,
    "source_split_sha256": EXPECTED_SPLIT_SHA,
    "strategy": "full",
    "precision": "bfloat16",
    "store_dtype": "float16",
    "block_size_for_roundtrip_decode": [128, 128, 128],
    "halo_for_roundtrip_decode": [16, 16, 16],
}

existing_payloads = list(bank_dir.glob("*/*.pt")) if bank_dir.exists() else []
if sentinel.exists():
    assert json.loads(sentinel.read_text(encoding="utf-8")) == request, (
        "STOP: the full-bank sentinel does not match this protocol."
    )
else:
    assert not existing_payloads, (
        "STOP: latent payloads exist without the expected full-bank sentinel. "
        "Do not mix or overwrite them."
    )
    bank_dir.mkdir(parents=True, exist_ok=True)
    temporary = sentinel.with_suffix(".tmp")
    temporary.write_text(
        json.dumps(request, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    temporary.replace(sentinel)

print("FULL BANK:", LATENTS)
print("FULL GATE WORK:", WORK)
print("Immutable identities and full-bank sentinel verified.")

## 3. Remap the frozen split to Drive paths

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "/content/MRIxFields/src")
from fieldbridge.data.manifests import record_from_mapping
from fieldbridge.data.vae_splits import VaeSplits, load_vae_splits, save_vae_splits

raw = json.loads(Path(SPLIT_SRC).read_text(encoding="utf-8"))
old = OLD_DATA_ROOT.replace("\\", "/").rstrip("/")

def remap(path):
    normalized = str(path).replace("\\", "/")
    return normalized.replace(old, DATA_ROOT.rstrip("/"))

missing = []
for split_name in ("train", "validation", "test"):
    assert len(raw["splits"][split_name]) == EXPECTED_COUNTS[split_name]
    for record in raw["splits"][split_name]:
        record["image_path"] = remap(record["image_path"])
        if not Path(record["image_path"]).is_file():
            missing.append(record["image_path"])

print("Missing remapped records:", len(missing))
if missing:
    print("First missing paths:", missing[:5])
assert not missing, "STOP: DATA_ROOT/remapping is wrong."

splits = VaeSplits(
    train=tuple(record_from_mapping(r) for r in raw["splits"]["train"]),
    validation=tuple(record_from_mapping(r) for r in raw["splits"]["validation"]),
    test=tuple(record_from_mapping(r) for r in raw["splits"]["test"]),
    seed=int(raw["seed"]),
    fractions=tuple(float(x) for x in raw["fractions"]),
    metadata=dict(raw.get("metadata", {})),
)
save_vae_splits(splits, SPLIT)
checked = load_vae_splits(SPLIT)
actual_counts = {
    "train": len(checked.train),
    "validation": len(checked.validation),
    "test": len(checked.test),
}
print("Remapped split counts:", actual_counts)
assert actual_counts == EXPECTED_COUNTS

## 3b. Resplit: move traveller 0006 into validation

`0006` and `0007` both sit in `Training_prospective`; `0009` is the only traveller in
`Testing_prospective`, and Validation has none. Leaving both training travellers in train
means the gate has no clean anchor — with `coupling: nn` a training traveller retrieves its
own target-field volume as its nearest neighbour, so scoring it measures memorization.

This moves `0006` to validation (1-1-1: 0007 train / 0006 validation / 0009 test). It is a
pure record relocation on the split JSON — no re-encode, no bank rebuild, since Stage-2 now
reads split membership from `--split-json` rather than from the bank manifest (a resplit
used to be a silent no-op for everything that reads the bank; it no longer is).


In [ ]:
from fieldbridge.data.resplit import resplit_file

SPLIT_RESPLIT = f"{WORK}/split_v4_111.json"

summary = resplit_file(SPLIT, SPLIT_RESPLIT, ["P:0006"], "validation")
print(json.dumps(summary, indent=2))
assert summary["counts"] == {"train": 1575, "validation": 204, "test": 205}, (
    "STOP: unexpected counts after the resplit -- check SPLIT is the remapped split_v3, "
    "not an already-resplit file."
)
print("\nSPLIT_RESPLIT ready:", SPLIT_RESPLIT)


## 4. Full-volume encode preflight

This deliberately tests `strategy="full"` on held-out **validation** records.
It never chooses tiled automatically and does not use the test split.

In [ ]:
import math
import time
import torch
from pathlib import Path
from fieldbridge.cli import _kl_vae_kwargs, _load_optional_config, _model_config
from fieldbridge.data.latent_bank import (
    decode_latent,
    downsample_factor,
    encode_latent,
    load_volume,
)
from fieldbridge.data.vae_splits import load_vae_splits
from fieldbridge.evaluation.metrics import ssim3d
from fieldbridge.models.factory import build_decoder, build_encoder
from fieldbridge.training.checkpoints import load_checkpoint

config = _load_optional_config(Path(VAE_CONFIG))
model_config = _model_config(config)
encoder = build_encoder("kl_vae", **_kl_vae_kwargs(model_config, "encoder"))
decoder = build_decoder("kl_vae", **_kl_vae_kwargs(model_config, "decoder"))
state = load_checkpoint(CKPT, map_location="cpu")
encoder.load_state_dict(state["encoder"])
decoder.load_state_dict(state["decoder"])
encoder.requires_grad_(False).eval().cuda()
decoder.requires_grad_(False).eval().cuda()
factor = downsample_factor(encoder)

validation_records = load_vae_splits(SPLIT).records_for("validation")[:2]
scores = []
started = time.time()

for record in validation_records:
    volume = load_volume(record).cuda()
    latent, strategy_used = encode_latent(
        encoder,
        volume,
        record.domain,
        strategy="full",
        block_size=(128, 128, 128),
        halo=(16, 16, 16),
        precision="bfloat16",
    )
    assert strategy_used == "full"
    reconstruction, decode_path = decode_latent(
        decoder,
        latent,
        record.domain,
        factor=factor,
        strategy="auto",
        block_size=(128, 128, 128),
        halo=(64, 64, 64),
        precision="bfloat16",
    )
    print(record.case_id, "decode path:", decode_path)
    score = float(ssim3d(reconstruction, volume, data_range=1.0))
    print(record.case_id, "full roundtrip SSIM3D:", score)
    assert math.isfinite(score)
    scores.append(score)
    del volume, latent, reconstruction
    torch.cuda.empty_cache()

mean_score = sum(scores) / len(scores)
seconds_per_volume = (time.time() - started) / len(scores)
print("Full preflight mean SSIM3D:", mean_score)
print("Approximate seconds/volume:", seconds_per_volume)
assert mean_score >= 0.95, "STOP: full encode roundtrip unexpectedly degraded."

encoder.cpu()
decoder.cpu()
del encoder, decoder, state
torch.cuda.empty_cache()
print("FULL-VOLUME PREFLIGHT PASSED")

## 5. Reuse the existing full-volume latent bank

The bank was built and verified in the run-C notebook against this exact checkpoint
(`74132b9c...`). `build-latent-bank` is idempotent, but on Drive a no-op pass still re-reads
every latent to recompute the stats, so it is skipped when the manifest is already present.
Delete the manifest to force a rebuild.


In [ ]:
import subprocess
from pathlib import Path

manifest_path = Path(LATENTS) / "latent_bank_manifest.json"

if manifest_path.is_file():
    print("BANK ALREADY BUILT, skipping:", manifest_path)
else:
    subprocess.run([
        "python", "-u", "-m", "fieldbridge.cli", "build-latent-bank",
        "--config", VAE_CONFIG,
        "--split-json", SPLIT,
        "--checkpoint", CKPT,
        "--out", LATENTS,
        "--device", "cuda",
        "--strategy", "full",
        "--block-size", "128", "128", "128",
        "--halo", "64", "64", "64",
        "--precision", "bfloat16",
        "--store-dtype", "float16",
        "--roundtrip-samples", "8",
    ], check=True)


## 6. Fail-closed full-bank verification

In [ ]:
import json
import math
import torch
from pathlib import Path

manifest_path = Path(LATENTS) / "latent_bank_manifest.json"
assert manifest_path.is_file(), (
    "Bank is incomplete (or the build was interrupted). Run all again to resume."
)
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

assert manifest["vae_checkpoint_sha256"] == EXPECTED_CKPT_SHA
assert manifest["config"]["strategy"] == "full"
assert manifest["config"]["precision"] == "bfloat16"
assert manifest["config"]["store_dtype"] == "float16"

totals = {
    split_name: int(manifest["counts"][split_name]["total"])
    for split_name in EXPECTED_COUNTS
}
assert totals == EXPECTED_COUNTS
assert len(manifest["records"]) == sum(EXPECTED_COUNTS.values()) == 1984

actual_files = {
    split_name: len(list((Path(LATENTS) / split_name).glob("*.pt")))
    for split_name in EXPECTED_COUNTS
}
print("Manifest totals:", totals)
print("Actual payload files:", actual_files)
assert actual_files == EXPECTED_COUNTS

for split_name in ("train", "validation", "test"):
    records = [r for r in manifest["records"] if r["split"] == split_name]
    for index in sorted({0, len(records) // 2, len(records) - 1}):
        payload_path = Path(LATENTS) / records[index]["path"]
        payload = torch.load(
            payload_path,
            map_location="cpu",
            weights_only=False,
        )
        assert payload["split"] == split_name
        assert payload["encode_strategy"] == "full"
        assert payload["encode_precision"] == "bfloat16"
        assert payload["vae_checkpoint_sha256"] == EXPECTED_CKPT_SHA
        print(
            split_name,
            payload_path.name,
            payload["encode_strategy"],
            payload["encode_precision"],
        )

roundtrip = float(manifest["roundtrip"]["mean_ssim3d"])
latent_std = float(manifest["latent_stats"]["global_std"])
assert math.isfinite(roundtrip) and roundtrip >= 0.95
assert math.isfinite(latent_std) and latent_std > 0
print("Roundtrip mean SSIM3D:", roundtrip)
print("Train latent global std:", latent_std)
print("FULL BANK VERIFIED: 1,984/1,984 payloads")

## 7. Build the NN-coupling descriptors (one time, cached to Drive)

The `nn` coupling needs one pooled, standardized vector per bank record. Building them reads
every latent once (~11 GB for train), so the result is cached on Drive and every later run —
FM, SB, resumes — hits the cache. The cache is fingerprinted by the record list and the pool
size, so a bank rebuild or a pool-size change invalidates it instead of silently returning
stale descriptors.


In [ ]:
import time
from pathlib import Path

from fieldbridge.data.latent_bank_dataset import LatentBankIndex, LatentStats
from fieldbridge.training.stage2_transport import build_latent_descriptors

# Write into the bank directory, which is exactly where training looks when the config leaves
# `descriptor_cache: null`. Building them somewhere else would silently pay the ~11 GB read
# twice: once here and once inside the training run.
POOL = 4  # must equal training.ot_pool_size in the v2 configs
stats = LatentStats.from_json(Path(LATENTS) / "latent_stats.json")

for split_name in ("train", "validation"):
    index = LatentBankIndex(LATENTS, split_name, split_json=SPLIT_RESPLIT)
    cache_path = Path(LATENTS) / f"descriptors_{split_name}_pool{POOL}.pt"
    started = time.time()
    vectors = build_latent_descriptors(
        index, stats, pool_size=POOL, cache_path=cache_path, log=True
    )
    assert vectors.shape[0] == len(index)
    assert bool(vectors.isfinite().all()), "STOP: non-finite descriptor."
    print(f"{split_name}: {tuple(vectors.shape)} in {time.time() - started:.0f}s -> {cache_path}")

# Sanity on the coupling itself: for a traveller, the globally nearest volume at another
# field should be that same subject. If this fails, the descriptors are not capturing anatomy
# and `nn` is no better than the random pairing it replaces. This index is built from
# SPLIT_RESPLIT, so it holds only 0007 -- 0006 was moved to validation and must NOT show up
# in the training coupling.
from fieldbridge.data.domains import Contrast
from fieldbridge.training.stage2_transport import _FieldPools

index = LatentBankIndex(LATENTS, "train", split_json=SPLIT_RESPLIT)
pools = _FieldPools.from_index(index).with_nn_tables(
    build_latent_descriptors(index, stats, pool_size=POOL,
                             cache_path=Path(LATENTS) / f"descriptors_train_pool{POOL}.pt"),
    candidates=1,
)
subject_of = {i: r.subject_id for i, r in enumerate(index.records)}
prospective = {i for i, r in enumerate(index.records) if str(r.case_id).startswith("P_")}
hits = total = 0
for (contrast, field_s, field_t), table in pools.nn_tables.items():
    for row, position in enumerate(pools.by_contrast_field[contrast][field_s]):
        if position not in prospective:
            continue
        total += 1
        hits += subject_of[int(table[row, 0])] == subject_of[position]
print(f"\ntraveller self-retrieval: {hits}/{total} nearest neighbours are the same subject")
print("DESCRIPTORS READY")

train_traveller_subjects = {subject_of[i] for i in prospective}
assert train_traveller_subjects == {"0007"}, (
    f"STOP: expected only traveller 0007 in the resplit train pool, got {train_traveller_subjects}. "
    "0006 leaking back into train would recontaminate the gate."
)
print("train traveller check OK: only", train_traveller_subjects, "in the coupling pool")


## 8. Sanity before the long run

`training-conventions` requires this and it has already paid for itself once today: the same
check on the Etapa-1 side caught a non-finite gradient that would have killed a 5-hour run.

Two things to read off this, both of which decide whether to continue:

* **the loss goes down.** 200 steps is not convergence, but a flat or rising curve is a bug.
* **the per-term breakdown.** `transport_cost` and `identity` are the two ladder terms turned
  on in v2, and their weights (0.1 each) are a *guess*, unlike the coupling change. If either
  term dwarfs `flow`, lower its weight here before spending GPU-hours — do not discover the
  imbalance at step 20,000.


In [ ]:
import re
import subprocess

SANITY_STEPS = 200
sanity_dir = Path(WORK) / "sanity_fm_v2"
sanity_dir.mkdir(parents=True, exist_ok=True)

completed = subprocess.run([
    "python", "-u", "-m", "fieldbridge.cli", "train-stage2-transport",
    "--config", "configs/experiment/stage2_transport_fm_v2.yaml",
    "--bank-dir", LATENTS,
    "--split-json", SPLIT_RESPLIT,
    "--checkpoint-dir", str(sanity_dir),
    "--steps", str(SANITY_STEPS),
    "--device", "cuda",
], check=True, capture_output=True, text=True)
log = completed.stdout + completed.stderr
print(log[-4000:])

steps = re.findall(r"step=(\d+)/\d+ loss=([0-9.]+) \[([^\]]*)\]", log)
assert steps, "STOP: no per-step lines parsed; check the log above."
first_loss, last_loss = float(steps[0][1]), float(steps[-1][1])
print(f"\nloss {first_loss:.4f} -> {last_loss:.4f}")

terms = dict(pair.split("=") for pair in steps[-1][2].split())
print("final per-term:", {k: float(v) for k, v in terms.items()})
flow = float(terms.get("flow", "nan"))
for name in ("transport_cost", "identity"):
    if name in terms:
        ratio = float(terms[name]) / flow if flow else float("inf")
        verdict = "OK" if ratio < 1.0 else "TOO LARGE -> lower its weight before the long run"
        print(f"  weighted {name}/flow = {ratio:.3f}  {verdict}")

assert last_loss < first_loss, "STOP: loss did not decrease over the sanity run."
print("\nSANITY PASSED")


## 9. FM v2 — recoverable long run

Writes an immutable checkpoint every 2,000 steps and a `_last` alias on clean completion.
Re-running the notebook resumes from the newest valid checkpoint and runs only the remainder.
Resume restores model and optimizer but not the sampler RNG, so an uninterrupted run is still
the strictest reproducibility path.


In [ ]:
import subprocess
from pathlib import Path
from fieldbridge.training.checkpoints import load_checkpoint

TARGET_STEPS = 20000
run_dir = Path(WORK) / "fm_v2" / "ckpt"
run_dir.mkdir(parents=True, exist_ok=True)

valid = []
for path in sorted(run_dir.glob("*.pt")):
    try:
        state = load_checkpoint(path, map_location="cpu")
        step = int(state.get("step", 0))
        if 0 <= step <= TARGET_STEPS:
            valid.append((step, path))
    except Exception as error:
        print("Ignoring invalid checkpoint:", path.name, repr(error))

start_step, resume_path = max(valid, key=lambda item: item[0]) if valid else (0, None)
print("fm_v2 resume from step:", start_step, resume_path)

if start_step < TARGET_STEPS:
    command = [
        "python", "-u", "-m", "fieldbridge.cli", "train-stage2-transport",
        "--config", "configs/experiment/stage2_transport_fm_v2.yaml",
        "--bank-dir", LATENTS,
        "--split-json", SPLIT_RESPLIT,
        "--checkpoint-dir", str(run_dir),
        "--steps", str(TARGET_STEPS - start_step),
        "--device", "cuda",
        "--json",
    ]
    if resume_path is not None:
        command += ["--resume-from", str(resume_path)]
    subprocess.run(command, check=True)
else:
    print("fm_v2 ALREADY COMPLETE")

last = run_dir / "transport_fm_ot_cfm_v2_last.pt"
assert last.is_file(), "Final checkpoint was not published."
print("fm_v2 final step:", int(load_checkpoint(last, map_location="cpu").get("step", 0)))


## 10. SB v2 — the primary rung

Identical to FM v2 except `bridge: schrodinger`, so FM vs SB stays a clean one-variable
comparison. Run it after FM so a session that dies partway still leaves one finished model.


In [ ]:
import subprocess
from pathlib import Path
from fieldbridge.training.checkpoints import load_checkpoint

TARGET_STEPS = 20000
run_dir = Path(WORK) / "sb_v2" / "ckpt"
run_dir.mkdir(parents=True, exist_ok=True)

valid = []
for path in sorted(run_dir.glob("*.pt")):
    try:
        state = load_checkpoint(path, map_location="cpu")
        step = int(state.get("step", 0))
        if 0 <= step <= TARGET_STEPS:
            valid.append((step, path))
    except Exception as error:
        print("Ignoring invalid checkpoint:", path.name, repr(error))

start_step, resume_path = max(valid, key=lambda item: item[0]) if valid else (0, None)
print("sb_v2 resume from step:", start_step, resume_path)

if start_step < TARGET_STEPS:
    command = [
        "python", "-u", "-m", "fieldbridge.cli", "train-stage2-transport",
        "--config", "configs/experiment/stage2_transport_sb_v2.yaml",
        "--bank-dir", LATENTS,
        "--split-json", SPLIT_RESPLIT,
        "--checkpoint-dir", str(run_dir),
        "--steps", str(TARGET_STEPS - start_step),
        "--device", "cuda",
        "--json",
    ]
    if resume_path is not None:
        command += ["--resume-from", str(resume_path)]
    subprocess.run(command, check=True)
else:
    print("sb_v2 ALREADY COMPLETE")

last = run_dir / "transport_sb_brownian_v2_last.pt"
assert last.is_file(), "Final checkpoint was not published."
print("sb_v2 final step:", int(load_checkpoint(last, map_location="cpu").get("step", 0)))


## 11. Gate: held-out 0006 (primary) + training 0007 (labeled upper bound)

**0009 is not touched here.** It was spent once as held-out evidence and re-tuning against it
would destroy the only unbiased read we have.

After the 3b resplit, **0006 is in validation** — the transport trains on 0007 and the
1900+ retrospective volumes only, and never sees 0006. That is the number that decides
promotion. **0007 stays in train** and is reported separately, only behind
`--allow-training-subjects`, purely as an optimistic upper bound: with `coupling: nn` it
retrieves its own target-field volume as the nearest neighbour, so its score reflects
memorization as much as generalization. Do not promote on it.

What to look for, in order:

* **SSIM vs identity.** This is the one that mattered and did not move in v1 (0.8730 ->
  0.8755 against a 0.9573 ceiling). nRMSE alone can be won by getting the global brightness
  right, which is exactly what v1 did.
* **the per-pair split.** A gain concentrated on the catastrophic pairs is the v1 failure
  repeating. Look for gains on the pairs where identity is already reasonable.
* **T2w.** It regressed in v1 (5/20 nRMSE and 2/20 SSIM wins) because it is already
  scale-matched across fields. If T2w still regresses, the model is still doing photometry.
* **`decode.path_used`.** Must read `["full"]`. `["tiled"]` means it fell back on OOM and the
  numbers carry a ~0.04 nRMSE decode approximation.


In [ ]:
import json
import subprocess
from pathlib import Path

def run_gate(tag, variant, *, subjects, allow_training_subjects, out_name):
    checkpoint = Path(WORK) / tag / "ckpt" / f"transport_{variant}_last.pt"
    out = Path(WORK) / tag / out_name
    if not checkpoint.is_file():
        print("SKIP (no checkpoint):", tag)
        return None
    command = [
        "python", "-u", "-m", "fieldbridge.cli", "eval-stage2-transport",
        "--config", f"configs/experiment/stage2_transport_{'fm' if tag.startswith('fm') else 'sb'}_v2.yaml",
        "--bank-dir", LATENTS,
        "--split-json", SPLIT_RESPLIT,
        "--transport-checkpoint", str(checkpoint),
        "--vae-config", VAE_CONFIG,
        "--vae-checkpoint", CKPT,
        "--subjects", *subjects,
        "--solver", "heun",
        "--n-steps", "20",
        "--metrics", "ssim", "nrmse",
        "--out", str(out),
        "--device", "cuda",
    ]
    if allow_training_subjects:
        command.append("--allow-training-subjects")
    subprocess.run(command, check=True)
    return json.loads(out.read_text(encoding="utf-8"))


def report(label, result):
    print(f"\n=== {label} ({result['num_pairs']} pairs) ===")
    print("subject_splits:", result.get("subject_splits"))
    print("decode:", result.get("decode"))
    for method in ("transport", "identity", "ceiling"):
        print(f"  {method:9s}", {k: round(float(v), 6) for k, v in result["overall"][method].items()})

    pairs = result["pairs"]
    easy = [p for p in pairs if p["identity"]["nrmse"] <= 1.0]
    hard = [p for p in pairs if p["identity"]["nrmse"] > 1.0]
    for name, subset in (("identity nrmse <= 1.0", easy), ("identity nrmse  > 1.0", hard)):
        if not subset:
            continue
        d_nrmse = sum(p["transport"]["nrmse"] - p["identity"]["nrmse"] for p in subset) / len(subset)
        d_ssim = sum(p["transport"]["ssim"] - p["identity"]["ssim"] for p in subset) / len(subset)
        print(f"  {name}  n={len(subset):3d}  d_nrmse={d_nrmse:+.4f}  d_ssim={d_ssim:+.4f}")
    wins = sum(p["transport"]["ssim"] > p["identity"]["ssim"] for p in pairs)
    print(f"  SSIM wins vs identity: {wins}/{len(pairs)}")


for tag, variant in (("fm_v2", "fm_ot_cfm_v2"), ("sb_v2", "sb_brownian_v2")):
    # PRIMARY: 0006 is in validation after the resplit -- the transport never saw it during
    # training, so this is the number that decides promotion.
    clean = run_gate(tag, variant, subjects=["0006"], allow_training_subjects=False,
                      out_name="transport_eval_0006_clean.json")
    if clean is not None:
        report(f"{tag}: HELD-OUT traveller 0006 (clean)", clean)

    # OPTIONAL: 0007 is in train. This number is an explicit, labeled optimistic upper bound,
    # never a promotion criterion -- the CLI would otherwise refuse to compute it.
    upper = run_gate(tag, variant, subjects=["0007"], allow_training_subjects=True,
                      out_name="transport_eval_0007_upper_bound.json")
    if upper is not None:
        report(f"{tag}: TRAINING traveller 0007 (optimistic upper bound, NOT a promotion signal)", upper)


## Verdict

v2 is worth promoting only if **the HELD-OUT 0006 read shows SSIM moving against identity on
the pairs where identity is already reasonable**. The 0007 upper bound is context, not a
criterion — a gap between the two readings is itself informative (a large one means the
model leans on paired memorization more than on the coupling change). A repeat of v1 — nRMSE gains concentrated on the catastrophic pairs,
SSIM flat, T2w regressing — means the coupling change was not enough and the next lever is
supervision, not more steps.

Only after v2 clears this bar on 0006/0007 is it worth spending 0009 again.
